# P3 · Auditoría de la evaluación congelada

La configuración de P3 se congeló **antes** de mirar septiembre (prueba). Esta auditoría:

1. **Congela** (`modeling.p3.congelar`): reajusta el KMeans en entrenamiento a partir de la
   preparación revisada y comprueba que coincide con lo registrado en
   `references/p3_frozen.json` (hash de la preparación, 369 estaciones y hash de los
   centroides).
2. **Evalúa una sola vez** (`modeling.p3.evaluar_congelado`): escribe un marcador
   `EVALUACION_INICIADA.json` **antes** de leer septiembre, asigna septiembre a los centroides
   fijos y registra un run en MLflow. El marcador impide repetir o sobrescribir la evaluación.
3. **Reproduce** (`evaluar_congelado(reproduccion=True)`): en una carpeta separada, con Git
   limpio, repite el procedimiento sin ajustar nada y compara métricas (rtol=atol=1e-12) y
   SHA-256 de asignaciones contra `references/p3_frozen.json`
   (`evaluation.verificar_reproduccion`). Una discrepancia detiene la ejecución; nunca
   provoca ajustes.

## Requisitos y limitaciones

- **Evidencias históricas en la máquina.** Usa `config.P3_PREPARACION_PARQUET`,
  `P3_VALIDACION_DIR` y `P3_SENSIBILIDAD_DIR` (`reports/p3-*-20261018/`). No se versionan;
  existen en la máquina donde se generaron.
- **El hash de la preparación depende de la plataforma.** Una preparación regenerada en otra
  computadora puede tener bytes distintos (en macOS sale `7b11cc1f…` frente a `9f8411ab…` de
  la referencia). En ese caso `congelar` se detiene con
  `La preparacion cambio desde la revision`. **No** se debe cambiar la referencia para que pase.
- La reproducción exige `git status` limpio (`git_dirty=false`).
- `evaluar_congelado` adjunta `estabilidad_entre_cohortes.csv` de la carpeta de sensibilidad
  histórica; ese archivo no lo genera `evaluation.exportar_sensibilidad`.
- Equivalente desde la terminal: `python -m mibici.modeling.p3 [--solo-congelar | --reproducir]`.

In [ ]:
from mibici.config import P3_EVALUACION_CONGELADA_DIR, P3_PREPARACION_PARQUET, P3_REPRODUCCION_DIR
from mibici.modeling.p3 import congelar, evaluar_congelado

{
    "preparacion_historica_existe": P3_PREPARACION_PARQUET.exists(),
    "evaluacion_congelada_existe": P3_EVALUACION_CONGELADA_DIR.exists(),
    "reproduccion_existe": P3_REPRODUCCION_DIR.exists(),
}

## 1. Congelar

Solo si la evaluación congelada todavía no existe. Si ya existe, no se vuelve a congelar.

In [ ]:
if not P3_EVALUACION_CONGELADA_DIR.exists():
    congelar()

## 2. Evaluación única de septiembre

Se ejecuta una sola vez. Si el marcador ya existe, la función se detiene con
`Evaluacion ya iniciada: no se repite ni sobrescribe`; es el comportamiento esperado.

In [ ]:
evaluar_congelado()

## 3. Reproducción

Requiere un commit limpio. Escribe en `P3_REPRODUCCION_DIR` y compara contra
`references/p3_frozen.json`.

In [ ]:
evaluar_congelado(destino=P3_REPRODUCCION_DIR, reproduccion=True)